# 08 · Market briefing and scenario analysis

**Research question:** How might the supplied fictional economy affect each company's business model and financial resilience?

**Decision date:** 31 December 2025. This is a qualitative exposure map, not a return forecast or portfolio recommendation. Market conditions come from *Candidate_Pack.pdf*, “Market briefing”; company descriptions and management estimates come from *Company_Factsheets.pdf*, one page per ticker. All scenario mechanisms are analytical inferences unless labelled as direct evidence.

## ⭐ 1. Inputs and assumptions

The pack's next-year figures are scenario assumptions. “Exposure” percentages below describe different bases (sales, borrowing, loans or invested assets), so they must not be added or compared as a single score.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

macro = pd.DataFrame([
    ('Policy interest rate', '5.0%', '4.5–5.0%'),
    ('Consumer inflation', '3.4%', 'About 3%'),
    ('Economic growth', '1.0%', '0.5–1.5%'),
    ('Big household purchases', 'Down 4%', 'Flat to down 3%'),
    ('Funded transport budget', 'Up 8%', 'Up 6%'),
    ('Routine healthcare demand', 'Up 5%', 'Up 4%'),
    ('Essential technology budgets', 'Up 7%', 'Up 5%'),
    ('Energy costs', '15% below recent peak', 'Possible 20% disruption rise'),
], columns=['Indicator', 'Latest reading', 'Pack base case / stated risk'])
print('Fictional market briefing at 31 December 2025')
display(macro)

## ⭐ 2. Company exposure matrix

The percentages are transcribed from each company's factsheet management-estimate paragraph. Blank means **not supplied**, not zero exposure. Refinancing percentages are shares of borrowing due to reprice or refinance by 2027. Credit-sensitive percentages refer to loans for lenders and invested assets for Crown Mutual.

In [ ]:
exposure_rows = [
    ('NCL','Technology',80,None,None,None,20,None,'Data-centre electricity and security spending'),
    ('BFC','Technology',90,None,None,None,10,None,'Large-contract renewals; high earnings multiple'),
    ('MGS','Technology',65,None,None,None,15,None,'Customer hiring affects recruitment revenue'),
    ('FAI','Technology',10,15,None,None,85,None,'Product investment and possible dilution'),
    ('GVH','Healthcare',None,None,85,None,55,None,'New clinics require funding before utilisation rises'),
    ('HLM','Healthcare',10,None,75,None,20,None,'Hospital purchase timing and product concentration'),
    ('ORP','Healthcare',None,None,90,None,10,None,'Routine-care demand; modest growth'),
    ('VGB','Healthcare',None,None,None,None,20,None,'Clinical trial and cash-runway uncertainty'),
    ('HHM','Consumer',None,90,None,None,65,None,'Furniture purchases can be postponed'),
    ('QCK','Consumer',None,70,None,None,90,None,'Delivery economics and cash burn'),
    ('FFF','Consumer',None,10,None,None,20,None,'Ingredient-cost pressure on thin margins'),
    ('THL','Consumer',None,95,None,None,80,None,'Discretionary apparel purchases and weak cash generation'),
    ('APR','Industrials',None,None,None,85,20,None,'Large-project timing and cash collection'),
    ('VTF','Industrials',10,25,None,25,80,None,'Factory investment and order conversion'),
    ('CSE','Industrials',None,None,10,65,25,None,'Wage inflation versus contract prices'),
    ('HBL','Industrials',None,65,None,None,75,None,'Vehicle and warehouse spending'),
    ('CDB','Financials',None,None,None,None,None,45,'Loan arrears and bank capital cushion'),
    ('LNP','Financials',75,40,None,None,10,None,'Payment demand and growth expectations'),
    ('CRM','Financials',None,10,None,None,None,10,'Claims and insurance capital coverage'),
    ('BSL','Financials',None,None,None,None,None,85,'Loan arrears and bank capital cushion'),
]
exposures = pd.DataFrame(exposure_rows, columns=['Ticker','Sector','Essential tech sales %','Optional spending sales %','Routine care sales %','Infrastructure sales %','Borrowing reprices by 2027 %','Rate-sensitive loans/assets %','Business-model risk'])
assert len(exposures)==20 and exposures['Ticker'].is_unique
print('Factsheet management estimates; percentages use different bases')
display(exposures.fillna('—'))

### Valuation vulnerability and concentrated exposures

The 2025 P/E below is calculated from the approved workbook and 31 December close, then interpreted alongside the factsheets. A high multiple can leave less room for disappointing growth, but it does not predict a price decline. Negative earnings make P/E undefined. The two charts use separate panels because sales exposure and borrowing repricing have different denominators.


In [ ]:
from pathlib import Path
import sys
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.data_loader import load_cleaned_data
from src.research.valuation import valuation_snapshot
annual, prices = load_cleaned_data(project_root / 'data' / 'Cleaned')
valuation = valuation_snapshot(annual, prices)
valuation_context = exposures[['Ticker','Sector','Business-model risk']].merge(valuation[['ticker','pe_ratio','fcf_yield']], left_on='Ticker', right_on='ticker', validate='one_to_one').drop(columns='ticker')
valuation_context['Valuation question (inference)'] = valuation_context.pe_ratio.apply(lambda x: 'Earnings multiple above 30x: test growth expectations' if pd.notna(x) and x > 30 else ('P/E undefined: inspect losses and cash runway' if pd.isna(x) else 'Compare multiple with cash quality and sector peers'))
print('2025 valuation context: P/E in multiples; FCF yield in fractions (0.10 = 10%)')
display(valuation_context.round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, column, title in [
    (axes[0], 'Optional spending sales %', 'Optional spending share of sales'),
    (axes[1], 'Borrowing reprices by 2027 %', 'Borrowing due to reprice/refinance')]:
    data = exposures[['Ticker',column]].dropna().sort_values(column)
    ax.barh(data['Ticker'], data[column], color='#5379a3')
    ax.set(title=title, xlabel='Factsheet estimate (%)', xlim=(0, 100))
    ax.grid(axis='x', alpha=.2)
fig.tight_layout()
plt.show()

## 3. Sector transmission and financial resilience

The sector table identifies channels to investigate, not predicted winners. The factsheets' lender and insurer warnings make industrial debt ratios unsuitable for their financial resilience checks.

In [ ]:
sector_channels = pd.DataFrame([
 ('Technology','Essential IT budgets; customer retention','Energy and data-centre costs; valuation expectations','FCF margin, contract concentration, borrowing repricing'),
 ('Healthcare','Routine-care demand','Clinic funding, hospital purchase delays, trial risk','FCF, debt, product concentration'),
 ('Consumer','Everyday staples demand for FFF','Weak optional purchases; input costs; refinancing','FCF after capex, margins, debt'),
 ('Industrials','Funded transport work','Project delays and wage or energy costs','Backlog conversion, cash collection, FCF'),
 ('Financials','Rate relief for some borrowers','Credit losses; claims; valuation expectations','Overdue loans and capital cushion for lenders; claims ratio and capital cover for CRM'),
], columns=['Sector','Potential support','Potential pressure','Resilience evidence to review'])
print('Sector-level qualitative channels')
display(sector_channels)

## ⭐ 4. Three scenarios

**Direct evidence** is the stated briefing or a factsheet estimate. **Inference** describes a plausible route from that evidence to sales, margins, cash flow or valuation; no precise company impact is asserted.

In [ ]:
scenarios = pd.DataFrame([
 ('Base case','Rates 4.5–5.0%; slow growth; optional purchases flat to down 3%; funded transport +6%; routine care +4%; essential tech +5%','HHM, QCK, THL and HBL have high stated optional-demand exposure; APR and CSE have infrastructure exposure; NCL, BFC and ORP have essential/routine demand exposure','Optional demand may constrain revenue; funded work may support order flow; financing and operating costs still determine margins and cash flow'),
 ('Faster rate decline','Rates fall faster than the pack base case (hypothesis; no numerical path supplied)','FAI, QCK, THL, VTF, HBL and GVH have substantial borrowing repricing exposure; CDB and BSL have rate-sensitive borrowers','Refinancing pressure and some borrower stress could ease; lender interest income, credit outcomes and equity valuation could also change in different directions'),
 ('Adverse: project delays / energy rise','Public projects delayed and/or energy costs rise; pack notes a possible 20% energy disruption rise','APR and CSE have high infrastructure-sales exposure; NCL has data-centre electricity costs; FFF has ingredient-cost sensitivity; VTF has factory investment','Delayed revenue and cash collection could pressure industrial FCF; higher input costs could narrow margins if not passed through; high multiples may become harder to sustain'),
], columns=['Scenario','Direct briefing evidence / stated hypothesis','Relevant company evidence','Possible transmission — inference'])
print('Qualitative scenario comparison; no forecast returns')
display(scenarios)

## 5. What requires further investigation?

- Check contract timing and cash collection for **APR** and **CSE** before translating a transport budget into company revenue.
- Test whether **NCL** and **FFF** can pass energy and input-cost increases through without losing demand.
- For **CDB** and **BSL**, compare overdue loans and capital cushions before interpreting rate relief as lower credit risk.
- For **CRM**, inspect claims and capital cover; its invested-asset exposure is not a loan-book percentage.
- Compare valuations and cash generation for highly priced growth businesses before assessing scenario sensitivity.

Historical relationships do not guarantee future outcomes. The exercise ends here: no security ranking, portfolio weights or investment conclusion is made.